In [ ]:
!python3 --version

In [ ]:
!nvidia-smi

In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
import tensorflow as tf

from tensorflow import keras

print(tf.__version__)
print(sys.version_info)
for module in mpl, np, pd, sklearn, tf, keras:
    print(module.__name__, module.__version__)

2.2.0
sys.version_info(major=3, minor=6, micro=9, releaselevel='final', serial=0)
matplotlib 3.2.1
numpy 1.18.5
pandas 1.0.4
sklearn 0.23.1
tensorflow 2.2.0
tensorflow.keras 2.3.0-tf


In [ ]:
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"Num GPUs Available: {len(gpus)}")

if gpus:
    print(f"GPUs available: {gpus}")
    try:
        # 打印每个GPU的详细信息
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True) # 推荐设置，按需分配显存
            print(f"Details for {gpu.name}:")
            # 可以尝试执行一个小操作来确认
        print("GPU is available and TensorFlow can see it!")

        # 尝试一个简单的GPU运算
        print("\nAttempting a simple computation on GPU...")
        with tf.device('/GPU:0'): # 明确指定在第一个GPU上运行
            a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            b = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            c = tf.matmul(a, b)
        print("Matrix multiplication result from GPU:")
        print(c.numpy())
        print("If no errors occurred, GPU is working!")

    except RuntimeError as e:
        print(f"RuntimeError during GPU setup or test: {e}")
else:
    print("GPU not available to TensorFlow. TensorFlow will run on CPU.")

In [2]:
# tf.function and auto-graph.
#自己实现一下elu激活函数,如果scale不为1，那就是selu
def scaled_elu(z, scale=1.0, alpha=1.0):
    # z >= 0 ? scale * z : scale * alpha * tf.nn.elu(z)
    is_positive = tf.greater_equal(z, 0.0)
#     return scale * tf.where(is_positive, z, alpha * tf.nn.elu(z))
    return scale * tf.where(is_positive, z, alpha * (tf.math.exp(z)-1))

#运行一下，这还是py函数
print(scaled_elu(tf.constant(-3.)))
print(scaled_elu(tf.constant([-3., -2.5])))

#把py实现的函数变为图实现的函数
#scaled_elu_tf就是图
scaled_elu_tf = tf.function(scaled_elu)
print(scaled_elu_tf(tf.constant(-3.)))
print(scaled_elu_tf(tf.constant([-3., -2.5])))

#可以通过这种方式找回原来的py函数
print(scaled_elu_tf.python_function is scaled_elu)
print(scaled_elu)
print(scaled_elu_tf)#tf的函数的执行效率比较高

tf.Tensor(-0.95021296, shape=(), dtype=float32)
tf.Tensor([-0.95021296 -0.917915  ], shape=(2,), dtype=float32)
tf.Tensor(-0.95021296, shape=(), dtype=float32)
tf.Tensor([-0.95021296 -0.917915  ], shape=(2,), dtype=float32)
True
<function scaled_elu at 0x7f7f105329d8>


In [3]:
#我们来测试一下性能，100万个数
%timeit scaled_elu(tf.random.normal((1000, 1000)))
%timeit scaled_elu_tf(tf.random.normal((1000, 1000)))

The slowest run took 60.75 times longer than the fastest. This could mean that an intermediate result is being cached.
236 ms ± 301 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
The slowest run took 7.05 times longer than the fastest. This could mean that an intermediate result is being cached.
164 ms ± 57.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
# 1 + 1/2 + 1/2^2 + ... + 1/2^n
#加了@tf.function装饰后就变为图结果，但是输入类型上不会有变化
@tf.function
def converge_to_2(n_iters):
    total = tf.constant(0.)
    increment = tf.constant(1.)
    for _ in range(n_iters):
        total += increment
        increment /= 2.0
    return total
print(converge_to_2)
print(converge_to_2(20))

tf.Tensor(1.9999981, shape=(), dtype=float32)


In [5]:
#如何看tf的图的代码
def display_tf_code(func):
    code = tf.autograph.to_code(func)
    from IPython.display import display, Markdown
    display(Markdown('```python\n{}\n```'.format(code)))

In [6]:
#传普通py函数,返回的是tf图的代码
display_tf_code(scaled_elu)

```python
def tf__scaled_elu(z, scale=None, alpha=None):
    do_return = False
    retval_ = ag__.UndefinedReturnValue()
    with ag__.FunctionScope('scaled_elu', 'fscope', ag__.ConversionOptions(recursive=True, user_requested=True, optional_features=(), internal_convert_user_code=True)) as fscope:
        is_positive = ag__.converted_call(tf.greater_equal, (z, 0.0), None, fscope)
        try:
            do_return = True
            retval_ = fscope.mark_return_value((scale * ag__.converted_call(tf.where, (is_positive, z, (alpha * (ag__.converted_call(tf.math.exp, (z,), None, fscope) - 1))), None, fscope)))
        except:
            do_return = False
            raise
    (do_return,)
    return ag__.retval(retval_)

```

In [9]:
#这个的前提是去除converge_to_2的装饰
# 因为converge_to_2有@tf.function标注，去掉应该就没问题了。to_code函数的输入是module,
# class, method, function, traceback, frame, or code object。不能是tf function.
display_tf_code(converge_to_2.python_function)

```python
def tf__converge_to_2(n_iters):
    do_return = False
    retval_ = ag__.UndefinedReturnValue()
    with ag__.FunctionScope('converge_to_2', 'fscope', ag__.ConversionOptions(recursive=True, user_requested=True, optional_features=(), internal_convert_user_code=True)) as fscope:
        total = ag__.converted_call(tf.constant, (0.0,), None, fscope)
        increment = ag__.converted_call(tf.constant, (1.0,), None, fscope)

        def get_state():
            return (total, increment)

        def set_state(loop_vars):
            nonlocal total, increment
            (total, increment) = loop_vars

        def loop_body(itr):
            nonlocal total, increment
            _ = itr
            total += increment
            increment /= 2.0
        ag__.for_stmt(ag__.converted_call(range, (n_iters,), None, fscope), None, loop_body, get_state, set_state, ('total', 'increment'), {})
        try:
            do_return = True
            retval_ = fscope.mark_return_value(total)
        except:
            do_return = False
            raise
    (do_return,)
    return ag__.retval(retval_)

```

In [10]:
#tf要把变量定义在函数外面，不能放里边
var = tf.Variable(0.)

@tf.function
def add_21():
    return var.assign_add(21) # += 

print(add_21())

tf.Tensor(21.0, shape=(), dtype=float32)


In [12]:
#cube计算立方，py是泛型设计，我们通过input_signature加类型限制可以防止调错
# @tf.function(input_signature=[tf.TensorSpec([None], tf.int32, name='x')])
@tf.function
def cube(z):
    return tf.pow(z, 3)

try:
    print(cube(tf.constant([1., 2., 3.])))
except ValueError as ex:
    print(ex)

print('-'*50)
#这行没问题
print(cube(tf.constant([1, 2, 3])))
print(cube)

Python inputs incompatible with input_signature:
  inputs: (
    tf.Tensor([1. 2. 3.], shape=(3,), dtype=float32))
  input_signature: (
    TensorSpec(shape=(None,), dtype=tf.int32, name='x'))
--------------------------------------------------
tf.Tensor([ 1  8 27], shape=(3,), dtype=int32)


In [13]:
int('abc')

ValueError: invalid literal for int() with base 10: 'abc'

In [16]:
# @tf.function py func -> tf graph
# get_concrete_function -> 给tf.function add input signature -> SavedModel

cube_func_int32 = cube.get_concrete_function(
    tf.TensorSpec([None], tf.int32))
print(cube_func_int32)
print(cube)

try:
    print(cube_func_int32(tf.constant([1, 2, 3])))
except Exception as ex:
    print(ex)

tf.Tensor([ 1  8 27], shape=(3,), dtype=int32)


In [17]:
#我们只要看原来函数和新生成的是否一致
# print(cube_func_int32 is cube.get_concrete_function())
print(cube.get_concrete_function(
    tf.constant([1, 2, 3])))
print(cube_func_int32)
print(cube_func_int32 is cube.get_concrete_function(
    tf.constant([1, 2, 3])))

True


In [18]:
print(cube_func_int32)
cube_func_int32.graph

In [19]:
#看下图定义都有哪些操作,了解即可
cube_func_int32.graph.get_operations()

[<tf.Operation 'x' type=Placeholder>,
 <tf.Operation 'Pow/y' type=Const>,
 <tf.Operation 'Pow' type=Pow>,
 <tf.Operation 'Identity' type=Identity>]

In [20]:
pow_op = cube_func_int32.graph.get_operations()[0]
print(pow_op)

name: "x"
op: "Placeholder"
attr {
  key: "_user_specified_name"
  value {
    s: "x"
  }
}
attr {
  key: "dtype"
  value {
    type: DT_INT32
  }
}
attr {
  key: "shape"
  value {
    shape {
      dim {
        size: -1
      }
    }
  }
}



In [12]:
print(list(pow_op.inputs))
print('-'*50)
print(list(pow_op.outputs))


[]
--------------------------------------------------
[<tf.Tensor 'z:0' shape=(None,) dtype=int32>]


In [13]:
#Placeholder用来放输入的地方，2.0中不需要，图中依然保留了
cube_func_int32.graph.get_operation_by_name("z")

<tf.Operation 'z' type=Placeholder>

In [16]:
cube_func_int32.graph.get_tensor_by_name("z:0")

<tf.Tensor 'z:0' shape=(None,) dtype=int32>

In [34]:
#打印出来看看图信息
cube_func_int32.graph.as_graph_def()

node {
  name: "z"
  op: "Placeholder"
  attr {
    key: "_user_specified_name"
    value {
      s: "z"
    }
  }
  attr {
    key: "dtype"
    value {
      type: DT_INT32
    }
  }
  attr {
    key: "shape"
    value {
      shape {
        dim {
          size: -1
        }
      }
    }
  }
}
node {
  name: "Pow/y"
  op: "Const"
  attr {
    key: "dtype"
    value {
      type: DT_INT32
    }
  }
  attr {
    key: "value"
    value {
      tensor {
        dtype: DT_INT32
        tensor_shape {
        }
        int_val: 3
      }
    }
  }
}
node {
  name: "Pow"
  op: "Pow"
  input: "z"
  input: "Pow/y"
  attr {
    key: "T"
    value {
      type: DT_INT32
    }
  }
}
node {
  name: "Identity"
  op: "Identity"
  input: "Pow"
  attr {
    key: "T"
    value {
      type: DT_INT32
    }
  }
}
versions {
  producer: 175
}